# コース進入予測 v2 — 2025年データ分析

## 方針
- デフォルト: 枠番 = コース
- 例外1: 前付け傾向のある選手（内コースに入る傾向）
- 例外2: 6コースに出る傾向のある新人選手

## 1. セットアップ & データ読み込み

In [1]:
from pathlib import Path
import calendar
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

repo_root = Path.cwd()
if not (repo_root / 'data').exists():
    repo_root = repo_root.parent.parent

print(f'Repository root: {repo_root}')

Repository root: /sessions/jolly-awesome-ptolemy/mnt/boatracecsv.github.io


In [2]:
# 2025年の programs + previews を読み込み
records = []
year = '2025'

for month in range(1, 13):
    _, max_day = calendar.monthrange(int(year), month)
    for day in range(1, max_day + 1):
        ms, ds = f'{month:02d}', f'{day:02d}'
        prog_path = repo_root / 'data' / 'programs' / year / ms / f'{ds}.csv'
        prev_path = repo_root / 'data' / 'previews' / year / ms / f'{ds}.csv'
        
        if not (prog_path.exists() and prev_path.exists()):
            continue
        
        try:
            prog = pd.read_csv(prog_path)
            prev = pd.read_csv(prev_path)
        except Exception:
            continue
        
        # previews と programs を レースコード でマージ（レース名も含める）
        prog_cols = ['レースコード', 'レース名'] + [c for c in prog.columns if '登録番号' in c or '選手名' in c or '級別' in c or '年齢' in c]
        prog_cols = [c for c in prog_cols if c in prog.columns]
        merged = prev.merge(prog[prog_cols], on='レースコード', how='left')
        
        for _, row in merged.iterrows():
            race_name = row.get('レース名', '')
            for boat in range(1, 7):
                course = row.get(f'艇{boat}_コース')
                reg_col = f'{boat}枠_登録番号'
                name_col = f'{boat}枠_選手名'
                grade_col = f'{boat}枠_級別'
                age_col = f'{boat}枠_年齢'
                
                if pd.isna(course):
                    continue
                
                records.append({
                    'レースコード': row['レースコード'],
                    'レース場': row.get('レース場', ''),
                    'レース名': race_name if pd.notna(race_name) else '',
                    '艇番': boat,  # = 枠番
                    'コース': int(course),
                    '登録番号': row.get(reg_col),
                    '選手名': row.get(name_col, ''),
                    '級別': row.get(grade_col, ''),
                    '年齢': row.get(age_col),
                })

df = pd.DataFrame(records)
print(f'総レコード数: {len(df):,}')
print(f'ユニーク選手数: {df["登録番号"].nunique():,}')
print(f'ユニークレース数: {df["レースコード"].nunique():,}')
print(f'レース名に「進入固定」を含むレコード数: {df["レース名"].str.contains("進入固定", na=False).sum():,}')

総レコード数: 334,713
ユニーク選手数: 1,636
ユニークレース数: 54,947
レース名に「進入固定」を含むレコード数: 5,742


## 2. 全体統計: 枠番 vs コースの関係

In [3]:
# 枠番=コースの割合（全体）
df['枠番一致'] = df['艇番'] == df['コース']
overall_match = df['枠番一致'].mean()
print(f'全体: 枠番=コース の割合 = {overall_match:.4f} ({overall_match*100:.1f}%)')
print()

# 枠番ごと
print('枠番ごとの一致率:')
for boat in range(1, 7):
    sub = df[df['艇番'] == boat]
    match_rate = (sub['コース'] == boat).mean()
    print(f'  {boat}枠 → {boat}コース: {match_rate:.4f} ({match_rate*100:.1f}%) [n={len(sub):,}]')
print()

# 枠番 vs コースのクロス集計
print('枠番→コース クロス集計 (%)：')
ct = pd.crosstab(df['艇番'], df['コース'], normalize='index') * 100
print(ct.round(1))

全体: 枠番=コース の割合 = 0.9176 (91.8%)

枠番ごとの一致率:
  1枠 → 1コース: 0.9919 (99.2%) [n=55,803]
  2枠 → 2コース: 0.9487 (94.9%) [n=55,782]
  3枠 → 3コース: 0.9294 (92.9%) [n=55,802]
  4枠 → 4コース: 0.8988 (89.9%) [n=55,778]
  5枠 → 5コース: 0.8596 (86.0%) [n=55,755]
  6枠 → 6コース: 0.8774 (87.7%) [n=55,793]

枠番→コース クロス集計 (%)：
コース     1     2     3     4     5     6
艇番                                     
1    99.2   0.7   0.1   0.0   0.0   0.0
2     0.5  94.9   3.6   0.6   0.3   0.2
3     0.2   1.6  92.9   4.1   0.8   0.4
4     0.1   0.8   1.8  89.9   4.9   2.4
5     0.1   1.1   0.7   3.9  86.0   8.3
6     0.1   1.2   1.0   1.7   8.2  87.7


## 2.5. 「進入固定」レースの検証

レース名に「進入固定」が含まれるレースでは、ルール上必ず枠番=コースとなる。このドメイン知識をデータで検証する。

In [4]:
# 進入固定レースの検証
is_fixed = df['レース名'].str.contains('進入固定', na=False)
df_fixed = df[is_fixed]
df_not_fixed = df[~is_fixed]

print('=== 「進入固定」レースの統計 ===')
print(f'進入固定レコード数: {len(df_fixed):,} / {len(df):,} ({len(df_fixed)/len(df)*100:.1f}%)')
print(f'進入固定ユニークレース数: {df_fixed["レースコード"].nunique():,}')
print()

# 進入固定レースで枠番=コースかどうか検証
if len(df_fixed) > 0:
    fixed_match = (df_fixed['艇番'] == df_fixed['コース']).mean()
    fixed_mismatch = df_fixed[df_fixed['艇番'] != df_fixed['コース']]
    print(f'進入固定レースの枠番一致率: {fixed_match:.6f} ({fixed_match*100:.2f}%)')
    print(f'進入固定レースの枠番不一致数: {len(fixed_mismatch)}')
    
    if len(fixed_mismatch) > 0:
        print('\n不一致レコード（データ異常の可能性）:')
        print(fixed_mismatch[['レースコード', 'レース名', '艇番', 'コース', '選手名']].to_string(index=False))
    else:
        print('→ 全件で枠番=コースが確認されました（ドメイン知識通り）')

print()
# 進入固定を除外した場合の枠番一致率
if len(df_not_fixed) > 0:
    not_fixed_match = (df_not_fixed['艇番'] == df_not_fixed['コース']).mean()
    print(f'進入固定以外の枠番一致率: {not_fixed_match:.4f} ({not_fixed_match*100:.1f}%)')
    print(f'  （参考: 全体の枠番一致率: {(df["艇番"] == df["コース"]).mean():.4f}）')
    print(f'  → 進入固定を除くと一致率が {(df["枠番一致"].mean() - not_fixed_match)*100:.2f}pt 下がる')

print()
# レース名のサンプルを表示
print('=== 進入固定レースのレース名サンプル ===')
fixed_race_names = df_fixed['レース名'].unique()
for name in fixed_race_names[:20]:
    count = (df_fixed['レース名'] == name).sum() // 6  # 6艇で割ってレース数に
    print(f'  [{count:3d}R] {name}')

=== 「進入固定」レースの統計 ===
進入固定レコード数: 5,742 / 334,713 (1.7%)
進入固定ユニークレース数: 958

進入固定レースの枠番一致率: 0.956984 (95.70%)
進入固定レースの枠番不一致数: 247

不一致レコード（データ異常の可能性）:
      レースコード             レース名  艇番  コース  選手名
202501032008     一般 進入固定 Ｈ１８０   4    5 稗田聖也
202501032008     一般 進入固定 Ｈ１８０   5    4 矢野真梨
202501121608 ガチ勝゛ち８ 進入固定 Ｈ１８０   5    6 岩永節也
202501121608 ガチ勝゛ち８ 進入固定 Ｈ１８０   6    5 佐野優都
202501271608 ガチ勝゛ち８ 進入固定 Ｈ１８０   4    6 川島圭司
202501271608 ガチ勝゛ち８ 進入固定 Ｈ１８０   5    4 前田健太
202501271608 ガチ勝゛ち８ 進入固定 Ｈ１８０   6    5 長尾萌加
202501272105 進入固定戦隊 進入固定 Ｈ１２０   5    6 村松遥輝
202501272105 進入固定戦隊 進入固定 Ｈ１２０   6    5 津田陸翔
202501282105 進入固定戦隊 進入固定 Ｈ１２０   4    5 田川大貴
202501282105 進入固定戦隊 進入固定 Ｈ１２０   5    6 伊藤尚汰
202501282105 進入固定戦隊 進入固定 Ｈ１２０   6    4 宮本夏樹
202502022008     一般 進入固定 Ｈ１８０   5    6 神田達也
202502022008     一般 進入固定 Ｈ１８０   6    5 山本凱立
202502052105 進入固定戦隊 進入固定 Ｈ１２０   4    6 桑島和宏
202502052105 進入固定戦隊 進入固定 Ｈ１２０   5    4 大西 賢
202502052105 進入固定戦隊 進入固定 Ｈ１２０   6    5 本田 愛
202502072105 進入固定戦隊 進入固定 Ｈ１８０   3    4 森 照夫
202502072105 進入固

## 3. 枠番 ≠ コース のケースを分析

In [5]:
# 枠番と異なるコースに入ったケース
mismatch = df[~df['枠番一致']].copy()
print(f'枠番≠コース のレコード数: {len(mismatch):,} / {len(df):,} ({len(mismatch)/len(df)*100:.1f}%)')
print()

# どのパターンが多いか
mismatch['パターン'] = mismatch['艇番'].astype(str) + '枠→' + mismatch['コース'].astype(str) + 'コース'
pattern_counts = mismatch['パターン'].value_counts().head(20)
print('枠番≠コースの頻出パターン Top20:')
print(pattern_counts.to_string())

枠番≠コース のレコード数: 27,566 / 334,713 (8.2%)

枠番≠コースの頻出パターン Top20:
パターン
5枠→6コース    4613
6枠→5コース    4601
4枠→5コース    2754
3枠→4コース    2291
5枠→4コース    2156
2枠→3コース    2033
4枠→6コース    1353
4枠→3コース    1002
6枠→4コース     969
3枠→2コース     872
6枠→2コース     651
5枠→2コース     594
6枠→3コース     552
4枠→2コース     471
3枠→5コース     451
5枠→3コース     412
1枠→2コース     383
2枠→4コース     328
2枠→1コース     262
3枠→6コース     236


## 4. 前付け傾向のある選手を特定

「前付け」= 枠番よりも内側（小さい番号）のコースに進入する傾向

In [6]:
# 各選手の「前付け率」を計算
# 前付け = コース < 枠番（内側に入る）
df['前付け'] = df['コース'] < df['艇番']
df['外寄せ'] = df['コース'] > df['艇番']

player_stats = df.groupby(['登録番号', '選手名']).agg(
    出走数=('レースコード', 'count'),
    前付け数=('前付け', 'sum'),
    外寄せ数=('外寄せ', 'sum'),
    枠番一致数=('枠番一致', 'sum'),
    平均枠番=('艇番', 'mean'),
    平均コース=('コース', 'mean'),
).reset_index()

player_stats['前付け率'] = player_stats['前付け数'] / player_stats['出走数']
player_stats['外寄せ率'] = player_stats['外寄せ数'] / player_stats['出走数']
player_stats['枠番一致率'] = player_stats['枠番一致数'] / player_stats['出走数']
player_stats['コース差'] = player_stats['平均コース'] - player_stats['平均枠番']

# 一定以上の出走数がある選手のみ（信頼性確保）
min_races = 30
reliable = player_stats[player_stats['出走数'] >= min_races].copy()
print(f'出走数 >= {min_races} の選手数: {len(reliable):,}')
print()

# 前付け率が高い選手（前付け傾向）
print('=== 前付け率が高い選手 Top30 ===')
top_maemae = reliable.nlargest(30, '前付け率')
print(top_maemae[['登録番号', '選手名', '出走数', '前付け率', '枠番一致率', '平均枠番', '平均コース', 'コース差']].to_string(index=False))

出走数 >= 30 の選手数: 1,594

=== 前付け率が高い選手 Top30 ===
  登録番号  選手名  出走数     前付け率    枠番一致率     平均枠番    平均コース      コース差
3159.0 江口晃生  192 0.531250 0.468750 3.161458 1.791667 -1.369792
3542.0 清水紀克  160 0.512500 0.475000 3.243750 2.193750 -1.050000
3072.0 西田 靖  159 0.490566 0.484277 3.377358 2.327044 -1.050314
3473.0 石川真二  224 0.450893 0.531250 3.330357 2.183036 -1.147321
3946.0 赤岩善生  245 0.424490 0.563265 3.285714 2.302041 -0.983673
3024.0 西島義則  185 0.362162 0.616216 3.264865 2.259459 -1.005405
3623.0 深川真二  217 0.354839 0.626728 3.331797 2.364055 -0.967742
3257.0 田頭 実  111 0.342342 0.639640 3.396396 2.603604 -0.792793
3582.0 吉川昭男  224 0.330357 0.647321 3.348214 2.651786 -0.696429
4874.0 池田奈津  113 0.292035 0.690265 3.575221 2.884956 -0.690265
3532.0 柴田 光  225 0.280000 0.706667 3.186667 2.528889 -0.657778
3554.0 仲口博崇  250 0.276000 0.712000 3.264000 2.728000 -0.536000
3265.0 今村暢孝  129 0.271318 0.713178 3.240310 2.589147 -0.651163
4199.0 北川潤二  222 0.270270 0.720721 3.297297 2.684685 -0.612613
3933.0 山

In [7]:
# 前付け率の分布を確認
print('=== 前付け率の分布 ===')
for threshold in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    count = (reliable['前付け率'] >= threshold).sum()
    print(f'  前付け率 >= {threshold:.0%}: {count}人')

print()
print('=== 前付け率が10%以上の選手の詳細 ===')
maemae_players = reliable[reliable['前付け率'] >= 0.10].sort_values('前付け率', ascending=False)
print(f'{len(maemae_players)}人')
print()

# この選手たちの枠番別コース分布を確認
for _, p in maemae_players.head(10).iterrows():
    reg = p['登録番号']
    name = p['選手名']
    sub = df[df['登録番号'] == reg]
    print(f'\n--- {name} (No.{int(reg)}) 出走{int(p["出走数"])}回, 前付け率{p["前付け率"]:.1%} ---')
    ct = pd.crosstab(sub['艇番'], sub['コース'])
    print(ct.to_string())

=== 前付け率の分布 ===
  前付け率 >= 5%: 295人
  前付け率 >= 10%: 68人
  前付け率 >= 15%: 30人
  前付け率 >= 20%: 19人
  前付け率 >= 30%: 9人
  前付け率 >= 50%: 2人

=== 前付け率が10%以上の選手の詳細 ===
68人


--- 江口晃生 (No.3159) 出走192回, 前付け率53.1% ---
コース   1   2  3  4  5  6
艇番                     
1    44   0  0  0  0  0
2     5  31  0  0  0  0
3     6  22  5  0  0  0
4     9  17  0  3  0  0
5     5  15  1  0  3  0
6     8  14  0  0  0  4

--- 清水紀克 (No.3542) 出走160回, 前付け率51.2% ---
コース   1   2   3  4  5
艇番                   
1    27   1   0  0  0
2     1  33   0  0  0
3     0  22  10  0  0
4     0  13   4  3  1
5     0  17   4  3  3
6     0   6   8  3  1

--- 西田 靖 (No.3072) 出走159回, 前付け率49.1% ---
コース   1   2  3  4  5  6
艇番                     
1    21   0  0  0  0  0
2     3  29  2  1  0  0
3     4  19  8  0  0  0
4     3  10  4  8  1  0
5     0  16  3  1  7  0
6     0  12  0  3  0  4

--- 石川真二 (No.3473) 出走224回, 前付け率45.1% ---
コース   1   2   3  4  5   6
艇番                       
1    46   0   0  0  0   0
2     8  34   1  0  0   0
3     7  

## 5. 6コースに出る傾向のある新人選手を特定

新人 = B2級 or 若い選手（登録番号が大きい）で、枠番に関わらず6コースに押し出される傾向

In [8]:
# 6コースに入った割合を選手別に算出
df['6コース'] = df['コース'] == 6

player_6course = df.groupby(['登録番号', '選手名', '級別']).agg(
    出走数=('レースコード', 'count'),
    _6コース数=('6コース', 'sum'),
    平均枠番=('艇番', 'mean'),
    平均コース=('コース', 'mean'),
).reset_index()

player_6course['6コース率'] = player_6course['_6コース数'] / player_6course['出走数']
player_6course['外寄せ度'] = player_6course['平均コース'] - player_6course['平均枠番']

# B2級の選手に絞って分析
b2_players = player_6course[player_6course['級別'] == 'B2'].copy()
print(f'B2級選手数: {len(b2_players)}')
print()

# B2以外との比較
for grade in ['A1', 'A2', 'B1', 'B2']:
    sub = player_6course[player_6course['級別'] == grade]
    if len(sub) > 0:
        avg_6rate = sub['6コース率'].mean()
        avg_shift = sub['外寄せ度'].mean()
        print(f'  {grade}: 平均6コース率={avg_6rate:.3f}, 平均外寄せ度={avg_shift:+.3f} ({len(sub)}人)')

B2級選手数: 261

  A1: 平均6コース率=0.142, 平均外寄せ度=-0.051 (408人)
  A2: 平均6コース率=0.152, 平均外寄せ度=-0.010 (506人)
  B1: 平均6コース率=0.153, 平均外寄せ度=+0.011 (947人)
  B2: 平均6コース率=0.407, 平均外寄せ度=+0.058 (261人)


In [9]:
# B2級で6コース率が高い選手
print('=== B2級: 6コース率が高い選手 ===')
b2_reliable = b2_players[b2_players['出走数'] >= 20].sort_values('6コース率', ascending=False)
print(b2_reliable.head(30)[['登録番号', '選手名', '出走数', '6コース率', '平均枠番', '平均コース', '外寄せ度']].to_string(index=False))

print()
print(f'\nB2級で6コース率 >= 30%: {(b2_reliable["6コース率"] >= 0.30).sum()}人')
print(f'B2級で6コース率 >= 40%: {(b2_reliable["6コース率"] >= 0.40).sum()}人')
print(f'B2級で6コース率 >= 50%: {(b2_reliable["6コース率"] >= 0.50).sum()}人')

=== B2級: 6コース率が高い選手 ===
  登録番号  選手名  出走数    6コース率     平均枠番    平均コース      外寄せ度
5441.0 鈴木雄登   28 1.000000 5.607143 6.000000  0.392857
5434.0 藤田康生   31 1.000000 5.548387 6.000000  0.451613
5433.0 米本圭佑   27 1.000000 5.777778 6.000000  0.222222
5432.0 安藤瑠希   23 1.000000 5.652174 6.000000  0.347826
5427.0 港 理樹   34 1.000000 5.735294 6.000000  0.264706
5399.0 長野未来   79 1.000000 5.708861 6.000000  0.291139
5419.0 櫻井 隼  107 1.000000 5.644860 6.000000  0.355140
5414.0 中澤英里   62 1.000000 5.677419 6.000000  0.322581
5402.0 塚越陸斗   92 0.989130 5.695652 5.989130  0.293478
5415.0 阿部未侑   86 0.988372 5.604651 5.988372  0.383721
5411.0 木下虎之   94 0.968085 5.723404 5.968085  0.244681
5410.0 今泉 澪   84 0.964286 5.678571 5.964286  0.285714
5440.0 薗頭 心   23 0.956522 6.000000 5.956522 -0.043478
5416.0 龍田真白   84 0.952381 5.642857 5.952381  0.309524
5394.0 福西広太   97 0.938144 5.639175 5.938144  0.298969
5393.0 藤原 駿   61 0.934426 5.770492 5.852459  0.081967
5379.0 廣部大明  135 0.911111 5.407407 5.777778  0.370370
5400

In [10]:
# 級別問わず、外寄せ傾向（枠番より外のコースに入る）が強い選手
print('=== 外寄せ度が高い選手 Top30（出走30回以上）===')
reliable_ext = player_6course[player_6course['出走数'] >= 30].copy()
top_outside = reliable_ext.nlargest(30, '外寄せ度')
print(top_outside[['登録番号', '選手名', '級別', '出走数', '6コース率', '平均枠番', '平均コース', '外寄せ度']].to_string(index=False))

=== 外寄せ度が高い選手 Top30（出走30回以上）===
  登録番号  選手名 級別  出走数    6コース率     平均枠番    平均コース     外寄せ度
5434.0 藤田康生 B2   31 1.000000 5.548387 6.000000 0.451613
4538.0 笠置博之 A2  180 0.244444 3.444444 3.872222 0.427778
3857.0 阿波勝哉 B2   70 0.257143 3.542857 3.928571 0.385714
5415.0 阿部未侑 B2   86 0.988372 5.604651 5.988372 0.383721
5405.0 竹内 来 B2   92 0.891304 5.500000 5.880435 0.380435
5379.0 廣部大明 B2  135 0.911111 5.407407 5.777778 0.370370
5419.0 櫻井 隼 B2  107 1.000000 5.644860 6.000000 0.355140
5368.0 宮崎心之 B2   80 0.900000 5.500000 5.850000 0.350000
5400.0 高村 諒 B2   82 0.902439 5.536585 5.878049 0.341463
5254.0 羽田妃希 B1   33 0.393939 4.000000 4.333333 0.333333
5357.0 田中瀬里 B2  134 0.574627 5.044776 5.373134 0.328358
4102.0 益田啓司 A2   86 0.220930 3.418605 3.744186 0.325581
5207.0 高山敬悟 B1   43 0.279070 3.511628 3.837209 0.325581
5414.0 中澤英里 B2   62 1.000000 5.677419 6.000000 0.322581
5381.0 有山 望 B2  103 0.650485 5.194175 5.514563 0.320388
4571.0 菅 章哉 A1  189 0.259259 3.412698 3.724868 0.312169
5416.0 龍田真白 B2  

## 6. レース場別の傾向

In [11]:
# レース場別の枠番一致率
STADIUM_NAMES = {
    1: '桐生', 2: '戸田', 3: '江戸川', 4: '平和島', 5: '多摩川', 6: '浜名湖',
    7: '蒲郡', 8: '常滑', 9: '津', 10: '三国', 11: 'びわこ', 12: '住之江',
    13: '尼崎', 14: '鳴門', 15: '丸亀', 16: '児島', 17: '宮島', 18: '徳山',
    19: '下関', 20: '若松', 21: '芦屋', 22: '福岡', 23: '唐津', 24: '大村'
}

# レース場番号を取得（文字列の場合と数値の場合がある）
df['レース場番号'] = df['レース場'].apply(lambda x: int(x) if str(x).isdigit() else None)

stadium_stats = df.groupby('レース場番号').agg(
    レース数=('レースコード', 'count'),
    枠番一致率=('枠番一致', 'mean'),
    前付け率=('前付け', 'mean'),
    外寄せ率=('外寄せ', 'mean'),
).reset_index()

stadium_stats['場名'] = stadium_stats['レース場番号'].map(STADIUM_NAMES)
stadium_stats = stadium_stats.sort_values('枠番一致率')

print('=== レース場別 枠番一致率 ===')
for _, row in stadium_stats.iterrows():
    bar = '█' * int(row['枠番一致率'] * 50)
    print(f"  場{int(row['レース場番号']):2d} {row['場名']:4s}: {row['枠番一致率']:.3f} ({row['枠番一致率']*100:.1f}%) 前付{row['前付け率']:.3f} 外寄{row['外寄せ率']:.3f} {bar}")

=== レース場別 枠番一致率 ===
  場 1 桐生  : 0.863 (86.3%) 前付0.063 外寄0.073 ███████████████████████████████████████████
  場23 唐津  : 0.882 (88.2%) 前付0.055 外寄0.064 ████████████████████████████████████████████
  場 6 浜名湖 : 0.892 (89.2%) 前付0.052 外寄0.056 ████████████████████████████████████████████
  場10 三国  : 0.895 (89.5%) 前付0.049 外寄0.057 ████████████████████████████████████████████
  場 7 蒲郡  : 0.902 (90.2%) 前付0.042 外寄0.056 █████████████████████████████████████████████
  場 8 常滑  : 0.902 (90.2%) 前付0.047 外寄0.051 █████████████████████████████████████████████
  場20 若松  : 0.903 (90.3%) 前付0.047 外寄0.050 █████████████████████████████████████████████
  場19 下関  : 0.906 (90.6%) 前付0.046 外寄0.048 █████████████████████████████████████████████
  場21 芦屋  : 0.911 (91.1%) 前付0.043 外寄0.046 █████████████████████████████████████████████
  場14 鳴門  : 0.916 (91.6%) 前付0.041 外寄0.043 █████████████████████████████████████████████
  場22 福岡  : 0.918 (91.8%) 前付0.034 外寄0.048 █████████████████████████████████████████████
  場12 住之江 : 0.919

## 7. まとめ: ルールベース予測の設計指針

In [12]:
# 最終サマリー
print('=' * 70)
print('コース進入予測 ルールベース設計のためのサマリー')
print('=' * 70)

print(f'\n■ 全体統計')
print(f'  枠番=コースの割合: {df["枠番一致"].mean():.4f} ({df["枠番一致"].mean()*100:.1f}%)')
print(f'  前付けの割合: {df["前付け"].mean():.4f} ({df["前付け"].mean()*100:.1f}%)')
print(f'  外寄せの割合: {df["外寄せ"].mean():.4f} ({df["外寄せ"].mean()*100:.1f}%)')

print(f'\n■ 前付け傾向選手（前付け率>=10%, 出走>=30回）')
maemae = reliable[reliable['前付け率'] >= 0.10]
print(f'  該当選手数: {len(maemae)}人')
print(f'  登録番号リスト: {sorted(maemae["登録番号"].astype(int).tolist())}')

print(f'\n■ 6コース傾向選手（6コース率>=30%, B2級, 出走>=20回）')
if len(b2_reliable) > 0:
    six_course = b2_reliable[b2_reliable['6コース率'] >= 0.30]
    print(f'  該当選手数: {len(six_course)}人')
    print(f'  登録番号リスト: {sorted(six_course["登録番号"].astype(int).tolist())}')

print(f'\n■ ベースライン精度（枠番=コースで予測した場合）')
print(f'  全体正解率: {df["枠番一致"].mean():.4f}')

# 例外処理した場合のシミュレーション
print(f'\n■ 例外処理で改善可能な件数')
print(f'  枠番≠コースの総数: {(~df["枠番一致"]).sum():,}')
maemae_regs = set(maemae['登録番号'].values)
mismatch_by_maemae = df[(~df['枠番一致']) & (df['登録番号'].isin(maemae_regs))]
print(f'  うち前付け選手によるもの: {len(mismatch_by_maemae):,}')
if len(b2_reliable) > 0 and len(six_course) > 0:
    six_regs = set(six_course['登録番号'].values)
    mismatch_by_six = df[(~df['枠番一致']) & (df['登録番号'].isin(six_regs))]
    print(f'  うち6コース傾向選手によるもの: {len(mismatch_by_six):,}')

コース進入予測 ルールベース設計のためのサマリー

■ 全体統計
  枠番=コースの割合: 0.9176 (91.8%)
  前付けの割合: 0.0383 (3.8%)
  外寄せの割合: 0.0441 (4.4%)

■ 前付け傾向選手（前付け率>=10%, 出走>=30回）
  該当選手数: 68人
  登録番号リスト: [3024, 3070, 3072, 3156, 3159, 3161, 3257, 3265, 3268, 3284, 3297, 3327, 3349, 3362, 3388, 3415, 3422, 3473, 3502, 3507, 3517, 3532, 3542, 3554, 3558, 3563, 3568, 3574, 3582, 3612, 3623, 3627, 3687, 3731, 3743, 3744, 3779, 3792, 3809, 3845, 3849, 3920, 3933, 3946, 4012, 4049, 4151, 4199, 4218, 4238, 4301, 4324, 4362, 4561, 4581, 4778, 4874, 4908, 5039, 5328, 5382, 5387, 5388, 5398, 5403, 5408, 5413, 5420]

■ 6コース傾向選手（6コース率>=30%, B2級, 出走>=20回）
  該当選手数: 98人
  登録番号リスト: [5230, 5302, 5304, 5309, 5314, 5318, 5326, 5327, 5328, 5336, 5337, 5338, 5340, 5341, 5342, 5343, 5344, 5345, 5346, 5347, 5348, 5349, 5350, 5351, 5352, 5353, 5354, 5355, 5357, 5358, 5359, 5360, 5361, 5362, 5363, 5364, 5365, 5366, 5367, 5368, 5369, 5370, 5371, 5372, 5373, 5374, 5375, 5376, 5377, 5378, 5379, 5380, 5381, 5382, 5383, 5384, 5385, 5386, 5387, 5388, 5389

## 8. ルールベース予測モデル

### ルール（優先順位順）
1. **進入固定ルール**: レース名に「進入固定」を含む場合 → コース = 枠番（完全一致を保証）
2. **前付け傾向**: 前付け率が高い選手がレースにいる場合 → その選手を内コースへ、他の選手を外へ押し出す
3. **6コース傾向**: B2級で6コース率が高い新人選手 → 6コースへ押し出す
4. **デフォルト**: コース = 枠番

In [13]:
# === 選手傾向テーブルの構築 ===
# 前付けルール: 無効化
maemae_set = set()
maemae_course_map = {}
print(f'前付け傾向選手数: 0 (前付けルール無効)')

# 6コース傾向選手: B2級, 6コース率>=80%, 出走回数制限なし
SIX_COURSE_THRESHOLD = 0.80
six_course_all = player_6course[(player_6course['級別'] == 'B2') & (player_6course['6コース率'] >= SIX_COURSE_THRESHOLD)]
six_course_set = set(six_course_all['登録番号'].values)
print(f'6コース傾向選手数（6コース率>={SIX_COURSE_THRESHOLD:.0%}）: {len(six_course_set)}')

# 枠番固定レース場
FIXED_STADIUM = {3}  # 江戸川
print(f'枠番固定レース場: {FIXED_STADIUM} (江戸川)')

print(f'\n閾値設定: 前付けルール=無効, 6コース率>={SIX_COURSE_THRESHOLD:.0%}, 出走回数制限なし')
print(f'特殊ルール: 江戸川（場3）は枠番=コース固定')


def predict_course_for_race(race_rows):
    """
    1レース分（6行）のデータからコースを予測する。
    
    ルール:
    1. 進入固定 → コース = 枠番（無条件）
    2. 江戸川（場3）→ コース = 枠番（固定）
    3. 6コース選手 → 6コースに割り当て（6コース率80%以上のB2選手）
    4. デフォルト → コース = 枠番
    
    玉突き処理: 6コースで空いた枠を他選手が埋める
    """
    boats = sorted(race_rows['艇番'].tolist())
    race_name = race_rows['レース名'].iloc[0] if 'レース名' in race_rows.columns else ''
    stadium = race_rows['レース場'].iloc[0] if 'レース場' in race_rows.columns else ''
    
    # レース場番号を取得
    try:
        stadium_num = int(stadium)
    except (ValueError, TypeError):
        stadium_num = None
    
    # ルール1: 進入固定
    if isinstance(race_name, str) and '進入固定' in race_name:
        return {b: b for b in boats}
    
    # ルール2: 枠番固定レース場
    if stadium_num in FIXED_STADIUM:
        return {b: b for b in boats}
    
    # 各艇の選手情報
    reg_nums = {}
    for _, row in race_rows.iterrows():
        reg_nums[row['艇番']] = row.get('登録番号', None)
    
    # --- 例外選手の検出 ---
    # 6コース選手
    six_assignments = {}
    for boat, reg in reg_nums.items():
        if reg in six_course_set and boat != 6:
            six_assignments[boat] = 6
    
    # 例外なし → デフォルト
    if not six_assignments:
        return {b: b for b in boats}
    
    # --- コース割り当て ---
    assigned = {}
    used_courses = set()
    
    # Step 1: 6コース選手
    for boat in six_assignments:
        if boat not in assigned and 6 not in used_courses:
            assigned[boat] = 6
            used_courses.add(6)
    
    # Step 2: 残りを枠番順に
    remaining_boats = sorted([b for b in boats if b not in assigned])
    remaining_courses = sorted([c for c in range(1, 7) if c not in used_courses])
    
    for boat, course in zip(remaining_boats, remaining_courses):
        assigned[boat] = course
    
    return assigned


# テスト
print('\n予測関数の構築完了')

前付け傾向選手数: 0 (前付けルール無効)
6コース傾向選手数（6コース率>=80%）: 39
枠番固定レース場: {3} (江戸川)

閾値設定: 前付けルール=無効, 6コース率>=80%, 出走回数制限なし
特殊ルール: 江戸川（場3）は枠番=コース固定

予測関数の構築完了


### 2025年全データで予測精度を検証

In [14]:
# === 2025年全データでルールベース予測を評価 ===
print('=== ルールベース予測の評価 ===\n')

# レースごとに予測を実行
predictions = []
race_codes = df['レースコード'].unique()

for race_code in race_codes:
    race_data = df[df['レースコード'] == race_code]
    if len(race_data) != 6:
        continue  # 6艇揃わないレースはスキップ
    
    pred = predict_course_for_race(race_data)
    
    for _, row in race_data.iterrows():
        boat = row['艇番']
        predictions.append({
            'レースコード': race_code,
            'レース名': row.get('レース名', ''),
            '艇番': boat,
            '実コース': row['コース'],
            '予測コース': pred.get(boat, boat),
            '登録番号': row.get('登録番号'),
        })

pred_df = pd.DataFrame(predictions)
pred_df['正解'] = pred_df['実コース'] == pred_df['予測コース']

# --- 全体精度 ---
overall_acc = pred_df['正解'].mean()
print(f'全体正解率: {overall_acc:.4f} ({overall_acc*100:.2f}%)')
print(f'  サンプル数: {len(pred_df):,}')
print(f'  正解数: {pred_df["正解"].sum():,}')
print(f'  不正解数: {(~pred_df["正解"]).sum():,}')

# --- ベースライン（枠番=コース）との比較 ---
baseline_acc = (pred_df['実コース'] == pred_df['艇番']).mean()
print(f'\nベースライン（枠番=コース）: {baseline_acc:.4f} ({baseline_acc*100:.2f}%)')
print(f'ルールベース予測:           {overall_acc:.4f} ({overall_acc*100:.2f}%)')
print(f'改善幅: {(overall_acc - baseline_acc)*100:+.2f}pt')

# --- 進入固定 vs それ以外 ---
is_fixed = pred_df['レース名'].str.contains('進入固定', na=False)
print(f'\n--- 進入固定レース ---')
if is_fixed.sum() > 0:
    fixed_acc = pred_df[is_fixed]['正解'].mean()
    print(f'正解率: {fixed_acc:.4f} ({fixed_acc*100:.2f}%) [n={is_fixed.sum():,}]')

print(f'\n--- 進入固定以外 ---')
if (~is_fixed).sum() > 0:
    not_fixed_acc = pred_df[~is_fixed]['正解'].mean()
    not_fixed_baseline = (pred_df[~is_fixed]['実コース'] == pred_df[~is_fixed]['艇番']).mean()
    print(f'ベースライン: {not_fixed_baseline:.4f} ({not_fixed_baseline*100:.2f}%)')
    print(f'ルールベース: {not_fixed_acc:.4f} ({not_fixed_acc*100:.2f}%)')
    print(f'改善幅: {(not_fixed_acc - not_fixed_baseline)*100:+.2f}pt')

# --- 枠番ごとの精度 ---
print(f'\n--- 枠番ごとの正解率 ---')
for boat in range(1, 7):
    sub = pred_df[pred_df['艇番'] == boat]
    acc = sub['正解'].mean()
    base = (sub['実コース'] == sub['艇番']).mean()
    diff = (acc - base) * 100
    print(f'  {boat}枠: ベースライン{base:.3f} → ルールベース{acc:.3f} ({diff:+.2f}pt)')

=== ルールベース予測の評価 ===



全体正解率: 0.9262 (92.62%)
  サンプル数: 320,454
  正解数: 296,795
  不正解数: 23,659

ベースライン（枠番=コース）: 0.9235 (92.35%)
ルールベース予測:           0.9262 (92.62%)
改善幅: +0.26pt

--- 進入固定レース ---
正解率: 0.9601 (96.01%) [n=5,712]

--- 進入固定以外 ---
ベースライン: 0.9229 (92.29%)
ルールベース: 0.9256 (92.56%)
改善幅: +0.27pt

--- 枠番ごとの正解率 ---
  1枠: ベースライン0.993 → ルールベース0.993 (-0.00pt)
  2枠: ベースライン0.952 → ルールベース0.952 (-0.00pt)
  3枠: ベースライン0.934 → ルールベース0.934 (+0.00pt)
  4枠: ベースライン0.905 → ルールベース0.906 (+0.05pt)
  5枠: ベースライン0.868 → ルールベース0.876 (+0.82pt)
  6枠: ベースライン0.889 → ルールベース0.896 (+0.72pt)


In [15]:
# === 予測が外れたケースの分析 ===
print('=== 予測ミスの分析 ===\n')

errors = pred_df[~pred_df['正解']].copy()
errors['パターン'] = errors['艇番'].astype(int).astype(str) + '枠→予測' + errors['予測コース'].astype(int).astype(str) + '実際' + errors['実コース'].astype(int).astype(str)

# ルールベースが正解してベースラインが不正解のケース（改善）
improved = pred_df[(pred_df['正解']) & (pred_df['実コース'] != pred_df['艇番'])]
print(f'ルールベースが正解＆ベースラインが不正解（改善）: {len(improved):,}件')

# ルールベースが不正解でベースラインが正解のケース（悪化）
degraded = pred_df[(~pred_df['正解']) & (pred_df['実コース'] == pred_df['艇番'])]
print(f'ルールベースが不正解＆ベースラインが正解（悪化）: {len(degraded):,}件')

# 両方不正解
both_wrong = pred_df[(~pred_df['正解']) & (pred_df['実コース'] != pred_df['艇番'])]
print(f'両方不正解: {len(both_wrong):,}件')

print(f'\n純改善数: {len(improved) - len(degraded):+,}件')

# 悪化ケースの詳細
if len(degraded) > 0:
    print(f'\n--- 悪化ケースの内訳 ---')
    deg_maemae = degraded[degraded['登録番号'].isin(maemae_set)]
    deg_others = degraded[~degraded['登録番号'].isin(maemae_set)]
    print(f'  前付け選手自身が悪化: {len(deg_maemae)}件')
    print(f'  玉突きで他選手が悪化: {len(deg_others)}件')

# レース単位の正解率
print(f'\n--- レース単位の精度 ---')
race_acc = pred_df.groupby('レースコード')['正解'].mean()
print(f'全艇正解（6/6）のレース: {(race_acc == 1.0).sum():,} / {len(race_acc):,} ({(race_acc == 1.0).mean()*100:.1f}%)')
print(f'5艇以上正解のレース: {(race_acc >= 5/6).sum():,} / {len(race_acc):,} ({(race_acc >= 5/6).mean()*100:.1f}%)')
print(f'平均正解艇数/レース: {race_acc.mean()*6:.2f}/6')

=== 予測ミスの分析 ===

ルールベースが正解＆ベースラインが不正解（改善）: 1,193件
ルールベースが不正解＆ベースラインが正解（悪化）: 347件
両方不正解: 23,312件

純改善数: +846件

--- 悪化ケースの内訳 ---
  前付け選手自身が悪化: 0件
  玉突きで他選手が悪化: 347件

--- レース単位の精度 ---
全艇正解（6/6）のレース: 45,511 / 53,409 (85.2%)
5艇以上正解のレース: 45,511 / 53,409 (85.2%)
平均正解艇数/レース: 5.56/6


## 9. 結論: コース進入予測は「枠番=コース」とする

### 検証結果まとめ

2025年通年データ（約33.5万レコード）および2026年1-3月データ（約6.5万レコード）を用いた検証の結果、以下が判明した。

**ベースライン（枠番=コース）の精度は92%前後**と非常に高く、これを上回るルールベース予測の構築は困難であった。

### 試行した例外ルールと結果

| ルール | 学習→テスト | 改善幅 | 備考 |
|---|---|---|---|
| 前付け50%+6コース70%+江戸川固定 | 2025自己評価 | +0.44pt | 2025年内では最良だが過学習の疑い |
| 前付け無効+6コース80%+江戸川固定 | 2025自己評価 | +0.26pt | 安定だが改善小 |
| 前付け無効+6コース80%+江戸川固定 | 2026/1-2→2026/3 | -0.19pt | 短期学習では6コース選手の検出困難 |
| 前付け50%+江戸川固定 | 2026/1-2→2026/3 | -0.91pt | 短期学習で前付けが過剰検出 |
| 前付け80%+江戸川固定 | 2025→2026/1-3 | ±0.00pt | 適用対象2組のみ、発火せず |

### 結論

- 前付け・6コースの例外ルールは、学習データと同じ期間では改善が見られるが、**未来データへの汎化性が乏しい**
- 前付け選手のコース予測自体は当たっても、**玉突きで他選手の予測が崩れる**問題が構造的に存在する
- 短期学習では閾値を満たす選手が偶発的にヒットし、長期学習では適用対象がほぼゼロになる

**よって、コース進入予測は「枠番=コース」（全艇で枠番通りに進入すると予測）を採用する。**